# PANZONER: From Binary Hypo Annotations to Spatial Layers

## 0. Parameters and Setup

In [ ]:
from pathlib import Path
INPUT_FILE = Path("data/UKF275_T_1_binary_input.tsv")
OUTPUT_DIR = Path("results")
MIN_REGION_SIZE = 20
N_LAYERS = 6


In [ ]:
import os
import json
import hashlib
import platform
from importlib.metadata import version
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz, load_npz
from sklearn.neighbors import NearestNeighbors
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle
from IPython.display import display

# Register the actual Arial font. Do not silently substitute another family.
font_candidates = [os.environ.get("PANZONER_ARIAL_FONT", ""),
                   "/home/biotree/.local/share/fonts/msttcorefonts/Arial.ttf"]
for candidate in font_candidates:
    if candidate and Path(candidate).is_file():
        font_manager.fontManager.addfont(candidate)
        break
arial_path = font_manager.findfont("Arial", fallback_to_default=False)
assert font_manager.FontProperties(fname=arial_path).get_name() == "Arial"
mpl.rcParams.update({
    "font.family": "Arial", "font.sans-serif": ["Arial"], "font.size": 8,
    "axes.titlesize": 9, "legend.fontsize": 7, "pdf.fonttype": 42,
    "ps.fonttype": 42, "svg.fonttype": "none", "figure.dpi": 110,
    "axes.linewidth": 0.5, "savefig.dpi": 300,
})
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)
environment = {p: version(p) for p in ["numpy", "pandas", "scipy", "scikit-learn", "matplotlib"]}
environment.update(python=platform.python_version(), arial_font=arial_path)
(OUTPUT_DIR / "environment.json").write_text(json.dumps(environment, indent=2))
display(pd.Series({**environment, "arial_font": Path(arial_path).name},
                  name="Version / font").to_frame())

In [ ]:
# Fixed colors for region and layer labels.
LAYER_COLORS = {"Hypo_region": "#1f77b4", "layer1": "#ff7f0e", "layer2": "#2ca02c",
                "layer3": "#d63138", "layer4": "#9467bd", "layer5": "#8c564b", "layer6": "#e377c2"}
FIXED_COLORS = {"Unassigned": "#d9dde2", "Other": "#d9dde2", "Hypo": "#1f77b4",
                "border": "#1f77b4", "core": "#ff7f0e", "isolated": "#2ca02c",
                "not_Hypo": "#d9dde2", "peri_core": "#9467bd"}
SEED_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d63138", "#9467bd", "#8c564b", "#e377c2"]
history = {}
step_records = []

def normalized_labels(values):
    return pd.Series(values, dtype=object).fillna("Unassigned").astype(str).to_numpy()

def label_color(label):
    if label in FIXED_COLORS:
        return FIXED_COLORS[label]
    if label.startswith("layer"):
        return LAYER_COLORS[label.split("_")[0]]
    if label.startswith("Hypo_region_"):
        return SEED_COLORS[(int(label.rsplit("_", 1)[1]) - 1) % len(SEED_COLORS)]
    return "#555555"

def category_order(values):
    values = list(pd.unique(values))
    def key(s):
        if s in ["Unassigned", "Other", "not_Hypo"]:
            return (9, 0, s)
        if s.startswith("Hypo_region_"):
            return (0, int(s.rsplit("_", 1)[1]), s)
        if s.startswith("layer"):
            return (1, int(s.split("_")[0].replace("layer", "")), s)
        return (2, 0, s)
    return sorted(values, key=key)

def draw_map(ax, labels, zoom=False, edges=None):
    labels = normalized_labels(labels)
    # Background first, then colored annotations.
    for label in reversed(category_order(labels)):
        mask = labels == label
        ax.scatter(xy[mask, 0], xy[mask, 1], s=9 if zoom else 4,
                   c=label_color(label), linewidths=0, rasterized=False, zorder=2)
    if edges is not None:
        ax.add_collection(LineCollection(xy[edges], colors="#656b73", linewidths=0.35,
                                         alpha=0.45, zorder=1, rasterized=False))
    bounds = ZOOM if zoom else FULL
    ax.set_xlim(bounds[0], bounds[1])
    ax.set_ylim(bounds[3], bounds[2])  # Original image orientation: y increases downward.
    ax.set_aspect("equal")
    ax.axis("off")
    if not zoom:
        ax.add_patch(Rectangle((ZOOM[0], ZOOM[2]), ZOOM[1]-ZOOM[0], ZOOM[3]-ZOOM[2],
                               fill=False, lw=0.7, edgecolor="#333333", zorder=3))

def save_figure(fig, stem):
    for ext in ["png", "pdf", "svg"]:
        fig.savefig(FIG_DIR / f"{stem}.{ext}", bbox_inches="tight")
    display(fig)
    plt.close(fig)

def show_step(stem, title, labels, edges=None):
    labels = normalized_labels(labels)
    history[stem] = labels.copy()
    categories = category_order(labels)
    counts = pd.Series(labels).value_counts().reindex(categories)
    table = counts.rename_axis("Label").rename("Spots").to_frame()
    table["Fraction"] = table.Spots / len(labels)
    table.to_csv(OUTPUT_DIR / f"{stem}_counts.tsv", sep="\t")
    pd.DataFrame({"spot_id": spots.spot_id, "label": labels}).to_csv(
        OUTPUT_DIR / f"{stem}_labels.tsv", sep="\t", index=False)
    if step_records and stem not in ["03_cell_status", "04_seed_regions"]:
        previous = history[step_records[-1]["step"]]
        changes = int(np.sum(labels != previous))
    else:
        changes = None
    step_records.append({"step": stem, "title": title, "changed_labels_vs_previous": changes})
    fig, axes = plt.subplots(1, 2, figsize=(8.2, 4.8))
    fig.subplots_adjust(left=0.02, right=0.98, top=0.84, bottom=0.19, wspace=0.06)
    fig.suptitle(title, y=0.97, fontsize=11)
    draw_map(axes[0], labels)
    draw_map(axes[1], labels, zoom=True, edges=edges)
    axes[0].set_title("Spatial overview", fontsize=9)
    axes[1].set_title("Fixed neighborhood zoom", fontsize=9)
    handles = [Line2D([], [], marker="o", linestyle="none", markersize=5,
                     color=label_color(c), label=f"{c} (n={counts[c]})") for c in categories]
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.01),
               ncol=min(3, len(handles)), frameon=False)
    save_figure(fig, stem)
    display(table)

## 1. Load Binary Hypo Annotations

In [ ]:
spots = pd.read_csv(INPUT_FILE, sep="\t")
assert spots.columns.tolist() == ["spot_id", "x", "y", "is_hypo"]
assert spots.spot_id.is_unique and spots.spot_id.notna().all()
assert spots.is_hypo.isin([0, 1]).all()
xy = spots[["x", "y"]].to_numpy()
assert np.isfinite(xy).all() and not spots[["x", "y"]].duplicated().any()
is_hypo = spots.is_hypo.to_numpy(dtype=bool)
n = len(spots)
assert n > 6 and is_hypo.any()
display(spots.head())
display(pd.Series({"All spots": n, "Hypo": int(is_hypo.sum()),
                   "Other": int((~is_hypo).sum())}, name="Spots").to_frame())

# Fixed display bounds from coordinates; zoom bounds from Hypo-labeled spots.
span = np.ptp(xy, axis=0)
FULL = (xy[:, 0].min()-0.04*span[0], xy[:, 0].max()+0.04*span[0],
        xy[:, 1].min()-0.04*span[1], xy[:, 1].max()+0.04*span[1])
nearest = NearestNeighbors(n_neighbors=1).fit(xy).kneighbors()[0]
display_spacing = float(np.median(nearest))
hxy = xy[is_hypo]
pad = 6 * display_spacing
ZOOM = (max(FULL[0], hxy[:, 0].min()-pad), min(FULL[1], hxy[:, 0].max()+pad),
        max(FULL[2], hxy[:, 1].min()-pad), min(FULL[3], hxy[:, 1].max()+pad))
show_step("01_binary_input", "1. Input: Hypo / Other", np.where(is_hypo, "Hypo", "Other"))

## 2. Build the Six-Neighbor Spatial Graph

In [ ]:
model = NearestNeighbors(n_neighbors=6, radius=1, metric="euclidean").fit(xy)
distances, neighbor_ids = model.kneighbors()  # X=None excludes the query itself.
distance_cutoff = float(np.median(distances) * 1.3)
rows = np.repeat(np.arange(n), 6)
cols = neighbor_ids.ravel()
keep_edge = distances.ravel() < distance_cutoff
graph = csr_matrix((np.ones(int(keep_edge.sum()), dtype=np.float32),
                    (rows[keep_edge], cols[keep_edge])), shape=(n, n))
graph.setdiag(0)
graph.eliminate_zeros()
graph.sort_indices()
# Store the outgoing neighbor indices of each spot for region traversal.
neighbors = {i: set(graph.indices[graph.indptr[i]:graph.indptr[i+1]]) for i in range(n)}
degree = np.diff(graph.indptr)
assert graph.diagonal().sum() == 0 and degree.max() <= 6
save_npz(OUTPUT_DIR / "spatial_connectivities.npz", graph)
display(pd.Series({"Directed adjacency entries": graph.nnz,
                   "Distance cutoff (pixels)": distance_cutoff,
                   "Minimum degree": degree.min(), "Maximum degree": degree.max(),
                   "Asymmetric entries": (graph != graph.T).nnz}, name="Value").to_frame())
display(pd.Series(degree).value_counts().sort_index().rename_axis("Neighbors").rename("Spots").to_frame())
rr, cc = graph.nonzero()
display_edges = np.unique(np.sort(np.column_stack([rr, cc]), axis=1), axis=0)
show_step("02_spatial_graph", "2. Rebuild the six-neighbor spatial graph",
          np.where(is_hypo, "Hypo", "Other"), edges=display_edges)

## 3. Classify Spots by Hypo Neighbor Counts

In [ ]:
hypo_neighbor_count = np.asarray(graph @ is_hypo.astype(int)).ravel().astype(int)
status = np.full(n, "not_Hypo", dtype=object)
status[is_hypo] = "isolated"
status[is_hypo & np.isin(hypo_neighbor_count, [3, 4])] = "border"
status[is_hypo & (hypo_neighbor_count == 5)] = "peri_core"
status[is_hypo & (hypo_neighbor_count == 6)] = "core"
show_step("03_cell_status", "3. Classify Hypo spots by their Hypo neighbors", status)
display(pd.crosstab(pd.Series(hypo_neighbor_count, name="Hypo neighbors"),
                    pd.Series(np.where(is_hypo, "Hypo", "Other"), name="Input")))

## 4. Identify Connected Seed Regions

In [ ]:
region = np.full(n, None, dtype=object)
seed_mask = np.isin(status, ["core", "peri_core"])
region_count = 0
for i in range(n):
    if seed_mask[i] and region[i] is None:
        region_count += 1
        label = f"Hypo_region_{region_count}"
        stack = [i]
        while stack:
            current = stack.pop()
            if region[current] is None:
                region[current] = label
                for j in neighbors[current]:
                    if seed_mask[j] and region[j] is None:
                        stack.append(j)
show_step("04_seed_regions", "4. Connected core / peri-core seeds", region)

## 5. Expand into Border Spots

In [ ]:
def grow_in_place(labels, allowed_statuses):
    # Newly labeled spots may expand in the same pass, so region membership can depend on input row order.
    for i in range(n):
        if labels[i] is not None:
            for j in neighbors[i]:
                if labels[j] is None and status[j] in allowed_statuses:
                    labels[j] = labels[i]
    return labels

region = grow_in_place(region, {"border"})
show_step("05_add_border", "5. First growth pass: border spots", region)

## 6. Expand into Border and Isolated Spots

In [ ]:
region = grow_in_place(region, {"border", "isolated"})
assert np.all(is_hypo[np.array([x is not None for x in region])])
show_step("06_add_isolated", "6. Second growth pass: border + isolated", region)

## 7. Merge Adjacent Regions

In [ ]:
adjacent_regions = {label: set() for label in pd.unique(region) if label is not None}
for i in range(n):
    if region[i] is not None:
        for j in neighbors[i]:
            if region[j] is not None and region[j] != region[i]:
                adjacent_regions[region[j]].add(region[i])
groups = [[label] + list(others) for label, others in adjacent_regions.items() if others]
changed = True
while changed:
    changed = False
    for i in range(len(groups)):
        for j in range(i+1, len(groups)):
            if set(groups[i]) & set(groups[j]):
                groups[j] = sorted(set(groups[i]) | set(groups[j]))
                groups.pop(i)
                changed = True
                break
        if changed:
            break
merge_map = {member: group[0] for group in groups for member in group[1:]}
region = np.array([merge_map.get(label, label) for label in region], dtype=object)
display(pd.DataFrame(list(merge_map.items()), columns=["Previous region", "Merged region"]))
show_step("07_merge_regions", "7. Merge directly adjacent Hypo regions", region)

## 8. Remove Regions with Fewer Than 20 Spots

In [ ]:
region_sizes = pd.Series([x for x in region if x is not None]).value_counts()
small_regions = set(region_sizes[region_sizes < MIN_REGION_SIZE].index)
display(region_sizes.rename_axis("Region").rename("Spots before filtering").to_frame())
region = np.array([None if x in small_regions else x for x in region], dtype=object)
retained_region = region.copy()
show_step("08_retained_regions", f"8. Retain Hypo regions with at least {MIN_REGION_SIZE} spots", region)

## 9. Expand Outward Layers

In [ ]:
def expand_one_layer(labels, layer_number):
    source_prefix = "Hypo_region_" if layer_number == 1 else f"layer{layer_number-1}_"
    added = []
    for i in range(n):
        if labels[i] is not None and labels[i].startswith(source_prefix):
            base_region = labels[i] if layer_number == 1 else labels[i][len(source_prefix):]
            for j in neighbors[i]:
                if labels[j] is None and status[j] == "not_Hypo":
                    labels[j] = f"layer{layer_number}_{base_region}"
                    added.append(j)
    return labels, np.array(added, dtype=int)

layer_snapshots = []
layer_counts = []

## 9.1. Generate Layer 1

In [ ]:
if N_LAYERS >= 1:
    region, added = expand_one_layer(region, 1)
    layer_snapshots.append(region.copy())
    layer_counts.append({"Layer": "layer1", "New spots": len(added)})
    show_step("09_layer1", "9.1. Add layer 1", region)
    print("New spots in layer1:", len(added))

## 9.2. Generate Layer 2

In [ ]:
if N_LAYERS >= 2:
    region, added = expand_one_layer(region, 2)
    layer_snapshots.append(region.copy())
    layer_counts.append({"Layer": "layer2", "New spots": len(added)})
    show_step("09_layer2", "9.2. Add layer 2", region)
    print("New spots in layer2:", len(added))

## 9.3. Generate Layer 3

In [ ]:
if N_LAYERS >= 3:
    region, added = expand_one_layer(region, 3)
    layer_snapshots.append(region.copy())
    layer_counts.append({"Layer": "layer3", "New spots": len(added)})
    show_step("09_layer3", "9.3. Add layer 3", region)
    print("New spots in layer3:", len(added))

## 9.4. Generate Layer 4

In [ ]:
if N_LAYERS >= 4:
    region, added = expand_one_layer(region, 4)
    layer_snapshots.append(region.copy())
    layer_counts.append({"Layer": "layer4", "New spots": len(added)})
    show_step("09_layer4", "9.4. Add layer 4", region)
    print("New spots in layer4:", len(added))

## 9.5. Generate Layer 5

In [ ]:
if N_LAYERS >= 5:
    region, added = expand_one_layer(region, 5)
    layer_snapshots.append(region.copy())
    layer_counts.append({"Layer": "layer5", "New spots": len(added)})
    show_step("09_layer5", "9.5. Add layer 5", region)
    print("New spots in layer5:", len(added))

## 9.6. Generate Layer 6

In [ ]:
if N_LAYERS >= 6:
    region, added = expand_one_layer(region, 6)
    layer_snapshots.append(region.copy())
    layer_counts.append({"Layer": "layer6", "New spots": len(added)})
    show_step("09_layer6", "9.6. Add layer 6", region)
    print("New spots in layer6:", len(added))

## 10. Layer Progression and Final Annotations

In [ ]:
assert 1 <= N_LAYERS <= 6, "This educational notebook shows up to six explicit layer cells."
fig, axes = plt.subplots(2, 3, figsize=(9.6, 7.4))
fig.subplots_adjust(left=0.02, right=0.98, top=0.90, bottom=0.14, hspace=0.12, wspace=0.06)
for k, ax in enumerate(axes.flat):
    if k < len(layer_snapshots):
        draw_map(ax, layer_snapshots[k], zoom=True)
        ax.set_title(f"Layer {k+1} added (+{layer_counts[k]['New spots']} spots)")
    else:
        ax.axis("off")
fig.suptitle("PANZONER: step-by-step outward layers", y=0.97, fontsize=12)
handles = [Line2D([], [], marker="o", linestyle="none", color=c, markersize=5, label=l)
           for l, c in LAYER_COLORS.items()]
handles.append(Line2D([], [], marker="o", linestyle="none", color=FIXED_COLORS["Unassigned"],
                      markersize=5, label="Unassigned"))
fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.015))
save_figure(fig, "10_layer_progression")
display(pd.DataFrame(layer_counts))

final = spots.copy()
final["hypo_neighbors"] = hypo_neighbor_count
final["cell_status"] = status
final["Hypo_region"] = region
def layer_number(label):
    if label is None:
        return pd.NA
    if label.startswith("Hypo_region_"):
        return 0
    return int(label.split("_")[0].replace("layer", ""))
final["layer"] = pd.array([layer_number(x) for x in region], dtype="Int64")
final.to_csv(OUTPUT_DIR / "final_spot_annotations.tsv", sep="\t", index=False)
pd.DataFrame(step_records).to_csv(OUTPUT_DIR / "step_log.tsv", sep="\t", index=False)
display(final.head(10))
display(pd.Series(normalized_labels(region)).value_counts().rename_axis("Final label").rename("Spots").to_frame())

# Check layer connectivity.
for i, label in enumerate(region):
    if label is None:
        continue
    if label.startswith("Hypo_region_"):
        assert is_hypo[i]
    else:
        assert not is_hypo[i]
        k = layer_number(label)
        base = label.split("_", 1)[1]
        previous_label = base if k == 1 else f"layer{k-1}_{base}"
        # Find an incoming edge from a source in the immediately preceding layer.
        incoming = graph[:, i].nonzero()[0]
        assert any(region[j] == previous_label for j in incoming)
print("Layer connectivity checked. Annotations saved.")

## 11. Core and L1–L6 Map

In [ ]:
from matplotlib.collections import PatchCollection
from matplotlib.patches import Circle
with np.load("data/visual_background.npz") as visual:
    he_image = visual["image"]
    he_scale = float(visual["scale"])
    spot_diameter = float(visual["spot_diameter_fullres"])
style = json.loads(Path("data/plot_style.json").read_text())
fig = plt.figure(figsize=(6.23, 6.09))
ax = fig.add_axes([0, 0.095, 1, 0.905])
ax.imshow(he_image, zorder=0)
scaled_xy = xy * he_scale
radius = spot_diameter * he_scale * 1.3 / 2
for k, color in enumerate(style["palette"]):
    wanted = "Hypo_region_1" if k == 0 else f"layer{k}_Hypo_region_1"
    mask = region == wanted
    circles = [Circle(tuple(c), radius) for c in scaled_xy[mask]]
    ax.add_collection(PatchCollection(circles, facecolor=color, edgecolor="none", linewidths=0, zorder=2))
xmin, xmax, ymin, ymax = style["crop_hires"]
ax.set_xlim(xmin, xmax); ax.set_ylim(ymax, ymin)
ax.set_aspect("equal"); ax.axis("off")
legend = [Line2D([], [], marker="o", linestyle="none", color=c, markersize=7, label=l)
          for l, c in zip(style["labels"], style["palette"])]
fig.legend(handles=legend, loc="lower center", bbox_to_anchor=(0.5, 0.015), ncol=7,
           handlelength=0.7, handletextpad=0.25, columnspacing=0.7, frameon=False, fontsize=10)
save_figure(fig, "pan_region_and_layers")